# News Discovery System — Google Colab Launcher

This notebook runs the existing **Gradio analyst UI** end-to-end (ingestion → normalization → aggregation → timeline) using the repository workflow (`run_workflow`) without rewriting system logic.

Run the cells top-to-bottom.

## 1) Environment setup
- Detect Colab runtime
- Define project location
- Set deterministic plotting backend

In [ ]:
from pathlib import Path
import os
import platform

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
PROJECT_DIR = Path('/content/news-discovery-system')

os.environ['MPLBACKEND'] = 'Agg'

print('Python:', platform.python_version())
print('In Colab:', IN_COLAB)
print('Project directory:', PROJECT_DIR)

## 2) Dependency installation
This installs notebook runtime dependencies plus app dependencies needed by `gr_app.py`.

In [ ]:
!pip -q install --upgrade pip
!pip -q install gradio matplotlib pandas

## 3) Project loading
If the repository is not already present in `/content`, this cell clones it.

- Default URL is a placeholder; replace once with your real repo URL.
- If you opened this notebook inside a preloaded workspace and `gr_app.py` already exists, clone is skipped.

In [ ]:
import shutil

REPO_URL = 'https://github.com/<YOUR_ORG>/news-discovery-system.git'

if (PROJECT_DIR / 'gr_app.py').exists():
    print('Repository already available:', PROJECT_DIR)
else:
    if '<YOUR_ORG>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL, then rerun this cell.')

    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)

    !git clone -q {REPO_URL} {PROJECT_DIR}
    print('Cloned to:', PROJECT_DIR)

%cd /content/news-discovery-system

## 4) Optional patching (Colab safety checks)
This does **minimal, non-destructive runtime checks** only:
- verifies required files
- adds project root to `sys.path`
- smoke-tests imports of `build_app` and `run_workflow`

In [ ]:
import sys

required = [
    PROJECT_DIR / 'gr_app.py',
    PROJECT_DIR / 'src' / 'news_app' / 'workflow.py',
    PROJECT_DIR / 'config' / 'sources.json',
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from gr_app import build_app, run_ui_workflow
from src.news_app.workflow import run_workflow

print('Imports OK. Colab runtime is ready.')

## 5) Launch Gradio UI
- Starts Gradio with `share=True` for external access
- Use the public link shown in output
- UI includes topic input, date range, run button, and output panels

In [ ]:
demo = build_app()
demo.queue()
demo.launch(share=True, inline=False, debug=True)

## Analyst validation checklist
After opening the Gradio public URL:

1. Enter a **topic** (example: `semiconductor`).
2. Enter **start date** and **end date** in `YYYY-MM-DD` format (range <= 30 days, end date not in future).
3. Click **Run Workflow**.
4. Verify outputs:
   - **Run Metadata**: shows `run_id`, `started_at`, and input payload.
   - **Step 1: Ingestion Validation**: check `hits_count`, `source`, and `raw_hits`.
   - **Step 2: Normalization Validation**: verify `canonical_articles`, `valid_count`, `invalid_count`, and `validation_issues`.
   - **Step 3: Aggregation + Timeline Validation**: confirm `daily_counts` aligns with plotted timeline points.

### Correctness spot-check (manual)
- Pick one day in `daily_counts` (e.g., `2026-04-10`).
- Count how many canonical articles have `published_at` on that day.
- Confirm it matches that day’s `article_count` in aggregation and the relative chart height.